# Ascent, canonical analysis, and constrained allocation

Classical RSM moves on a fitted surface: the gradient and Hessian of `forward()` (jax when
available, unit-invariant finite differences otherwise), steepest ascent, canonical analysis of
the stationary point, and the budget-constrained allocator with its effort frontier.

In [ ]:
import numpy as np

from axiom.core import D, Outcome, Treatment, is_failure
from axiom.surface import (
    Allocation, AllocationMethod, AscentPath, Bounds, Frontier, HillKernel, Method, Objective,
    StationaryPoint, Surface, SurfaceSpec, allocate, canonical_analysis, frontier, gradient, hessian,
    steepest_ascent,
)

In [ ]:
spec = SurfaceSpec(
    name="two_hill",
    treatments=(Treatment(name="a", dimension=D.currency, unit="USD"), Treatment(name="b", dimension=D.currency, unit="USD")),
    outcome=Outcome(name="y", dimension=D.outcome),
    kernels={"a": HillKernel(reference_dose=50.0), "b": HillKernel(reference_dose=20.0)},
)
surface = Surface(spec)
theta = {"alpha": 1.0, "k_a": 50.0, "s_a": 2.0, "beta_a": 10.0, "k_b": 20.0, "s_b": 1.5, "beta_b": 5.0, "sigma": 1.0}
bounds = Bounds(treatments=("a", "b"), low=(0.0, 0.0), high=(200.0, 80.0))
method: Method = "auto"
print(gradient(surface, theta, {"a": 30.0, "b": 10.0}, method=method))
print(np.round(hessian(surface, theta, {"a": 30.0, "b": 10.0}, method=method), 5))

In [ ]:
path = steepest_ascent(surface, theta, {"a": 10.0, "b": 5.0}, step=5.0, n_steps=20, bounds=bounds)
if isinstance(path, AscentPath):
    print(path.n, path.stop, path.best(), round(path.values[-1], 3))
else:
    print(path)

A saturating surface has no interior maximum, so canonical analysis reports a ridge or a
`Unsupported`; on a quadratic it finds and classifies the stationary point.

In [ ]:
sp = canonical_analysis(surface, theta, {"a": 30.0, "b": 10.0}, method=method)
print(sp if is_failure(sp) else (sp.kind, sp.point, sp.eigenvalues))

## Allocation under a budget

`allocate` maximizes the expected outcome subject to a total-dose budget and box bounds
(SLSQP; the cvxpy path is optional). Non-convergence is a typed `Unsupported`, never a number —
the parent repo once shipped an allocator that swallowed SLSQP's failure flag.

In [ ]:
obj: Objective = "mean"
meth: AllocationMethod = "slsqp"
alloc = allocate(surface, theta, budget=60.0, bounds=bounds, objective=obj, method=meth, seed=0)
if isinstance(alloc, Allocation):
    print(alloc.doses, round(alloc.expected_outcome, 3), alloc.status, round(alloc.total_dose, 3))
else:
    print(alloc)
print(allocate(surface, theta, budget=60.0, bounds=bounds, maxiter=1))

In [ ]:
fr = frontier(surface, theta, budgets=[10.0, 30.0, 60.0, 120.0], bounds=bounds, seed=0)
if isinstance(fr, Frontier):
    print(fr.as_frame())
    print("shadow prices:", np.round(fr.shadow_prices(), 4))
else:
    print(fr)